##Setup

You will need to make a copy of this notebook in your Google Drive before you can edit the homework files. You can do so with **File &rarr; Save a copy in Drive**. Please run on GPU: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU

In [1]:
#@title mount your Google Drive
#@markdown Your work will be stored in a folder called `hw_16831` by default to prevent Colab instance timeouts from deleting your edits.

import os
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
#@title set up mount symlink

DRIVE_PATH = '/content/gdrive/My\ Drive/hw_16831'
DRIVE_PYTHON_PATH = DRIVE_PATH.replace('\\', '')
if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

## the space in `My Drive` causes some issues,
## make a symlink to avoid this
SYM_PATH = '/content/hw_16831'
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH

<>:3: SyntaxWarning: invalid escape sequence '\ '
<>:3: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-3425321108.py:3: SyntaxWarning: invalid escape sequence '\ '
  DRIVE_PATH = '/content/gdrive/My\ Drive/hw_16831'


In [3]:
#@title apt install requirements

#@markdown Run each section with Shift+Enter

#@markdown Double-click on section headers to show code.

!apt update
!apt install -y --no-install-recommends \
        build-essential \
        curl \
        git \
        gnupg2 \
        make \
        cmake \
        ffmpeg \
        swig \
        libz-dev \
        unzip \
        zlib1g-dev \
        libglfw3 \
        libglfw3-dev \
        libxrandr2 \
        libxinerama-dev \
        libxi6 \
        libxcursor-dev \
        libgl1-mesa-dev \
        libgl1-mesa-glx \
        libglew-dev \
        libosmesa6-dev \
        lsb-release \
        ack-grep \
        patchelf \
        wget \
        xpra \
        xserver-xorg-dev \
        xvfb \
        python3-opengl \
        ffmpeg

Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
105 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as r

In [4]:
#@title clone homework repo

%cd $SYM_PATH
!git clone https://github.com/cmuroboticsdrl/16831-S26-HW.git
%cd 16831-S26-HW/hw1
%pip install -r requirements_colab.txt
%pip install -e .

/content/gdrive/My Drive/hw_16831
fatal: destination path '16831-S26-HW' already exists and is not an empty directory.
/content/gdrive/My Drive/hw_16831/16831-S26-HW/hw1
Obtaining file:///content/gdrive/My%20Drive/hw_16831/16831-S26-HW/hw1
  Preparing metadata (setup.py) ... done
  Running setup.py develop for rob831


In [5]:
# downloads mujoco from source
!wget https://mujoco.org/download/mujoco210-linux-x86_64.tar.gz
!tar xzvf mujoco210-linux-x86_64.tar.gz
!mkdir -p ~/.mujoco
!mv mujoco210 ~/.mujoco/mujoco210
!rm mujoco*

# set env variables before installing mujoco-py
import os
os.environ['LD_LIBRARY_PATH'] += ':/root/.mujoco/mujoco210/bin'
os.environ['MUJOCO_PY_MUJOCO_PATH'] = '/root/.mujoco/mujoco210'
os.environ['LD_LIBRARY_PATH'] += ':/usr/lib/nvidia'

%pip install -U mujoco
%pip install -U 'mujoco-py<2.2,>=2.1'
%pip install -U pyvirtualdisplay
%pip install -U gym-notebook-wrapper
%pip install -U "cython<3"

!cp /root/.mujoco/mujoco210/bin/*.so /usr/lib/x86_64-linux-gnu/

--2026-02-05 02:27:39--  https://mujoco.org/download/mujoco210-linux-x86_64.tar.gz
Resolving mujoco.org (mujoco.org)... 216.239.38.21, 216.239.36.21, 216.239.34.21, ...
Connecting to mujoco.org (mujoco.org)|216.239.38.21|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz [following]
--2026-02-05 02:27:39--  https://github.com/google-deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/400501136/1f51148e-4e64-4a12-a400-d6f1e21be444?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-05T03%3A18%3A25Z&rscd=attachment%3B+filename%3Dmujoco210-linux-x86_64.tar.gz&rsct=application%2Foctet-stream&s

In [6]:
#@title set up virtual display

from pyvirtualdisplay import Display

display = Display(visible=0, size=(1400, 900))
display.start()

In [7]:
import numpy as np
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

In [8]:
#@title test virtual display

#@markdown If you see a video of a four-legged ant fumbling about, setup is complete!

import numpy as np
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

import gym
import gnwrapper

env = gnwrapper.LoopAnimation(gym.make('Ant-v2'))

observation = env.reset()
for i in range(100):
    obs, rew, term, _ = env.step(env.action_space.sample())
    env.render()
    if term:
      break

env.display()
env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment Ant-v2 is out of date. You should consider upgrading to version `v4`.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/Cython/Distutils/old_build_ext.py:15: DeprecationWarning: dep_util is Deprecated. Use functions from setuptools instead.
  from distutils.dep_util import newer, newer_group
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UT

Compiling /usr/local/lib/python3.12/dist-packages/mujoco_py/cymj.pyx because it depends on /usr/local/lib/python3.12/dist-packages/Cython/Includes/cpython/object.pxd.
[1/1] Cythonizing /usr/local/lib/python3.12/dist-packages/mujoco_py/cymj.pyx


KeyboardInterrupt: 

## Editing Code

To edit code, click the folder icon on the left menu. Navigate to the corresponding file (`hw_16831/...`). Double click a file to open an editor. There is a timeout of about ~12 hours with Colab while it is active (and less if you close your browser window). We sync your edits to Google Drive so that you won't lose your work in the event of an instance timeout, but you will need to re-mount your Google Drive and re-install packages with every new instance.

**IMPORTANT**: Please re-run the following cell before every code run.

In [22]:
#@title imports
#@markdown Please re-run this cell before every code run

import os
import time
import importlib

import rob831.policies.MLP_policy
import rob831.policies.loaded_gaussian_policy
import rob831.agents.bc_agent
import rob831.infrastructure.rl_trainer
import rob831.infrastructure.utils

importlib.reload(rob831.policies.MLP_policy)
importlib.reload(rob831.policies.loaded_gaussian_policy)
importlib.reload(rob831.agents.bc_agent)
importlib.reload(rob831.infrastructure.rl_trainer)
importlib.reload(rob831.infrastructure.utils)

from rob831.infrastructure.rl_trainer import RL_Trainer
from rob831.agents.bc_agent import BCAgent
from rob831.policies.loaded_gaussian_policy import LoadedGaussianPolicy

## Run Behavior Cloning (Problem 1)

In [23]:
#@title runtime arguments

class Args:

  def __getitem__(self, key):
    return getattr(self, key)

  def __setitem__(self, key, val):
    setattr(self, key, val)

  #@markdown expert data
  expert_policy_file = '/content/hw_16831/16831-S26-HW/hw1/rob831/policies/experts/Humanoid.pkl' #@param
  expert_data = '/content/hw_16831/16831-S26-HW/hw1/rob831/expert_data/expert_data_Humanoid-v2.pkl' #@param
  env_name = 'Humanoid-v2' #@param ['Ant-v2', 'Humanoid-v2', 'Walker2d-v2', 'HalfCheetah-v2', 'Hopper-v2']
  exp_name = 'test_bc_humanoid_dagger' #@param
  do_dagger = True #@param {type: "boolean"}
  ep_len = 1000 #@param {type: "integer"}
  save_params = False #@param {type: "boolean"}

  num_agent_train_steps_per_iter = 5000 #@param {type: "integer"})
  n_iter = 100 #@param {type: "integer"})

  #@markdown batches & buffers
  batch_size = 1000 #@param {type: "integer"})
  eval_batch_size = 10000 #@param {type: "integer"}
  train_batch_size = 100 #@param {type: "integer"}
  max_replay_buffer_size = 1000000 #@param {type: "integer"}

  #@markdown network
  n_layers = 2 #@param {type: "integer"}
  size = 64 #@param {type: "integer"}
  learning_rate = 5e-3 #@param {type: "number"}

  #@markdown logging
  video_log_freq = -1 #@param {type: "integer"}
  scalar_log_freq = 1 #@param {type: "integer"}

  #@markdown gpu & run-time settings
  no_gpu = False #@param {type: "boolean"}
  which_gpu = 0 #@param {type: "integer"}
  seed = 1 #@param {type: "integer"}

args = Args()


In [24]:
#@title define `BC_Trainer`
class BC_Trainer(object):

    def __init__(self, params):
        #######################
        ## AGENT PARAMS
        #######################

        agent_params = {
            'n_layers': params['n_layers'],
            'size': params['size'],
            'learning_rate': params['learning_rate'],
            'max_replay_buffer_size': params['max_replay_buffer_size'],
            }

        self.params = params
        self.params['agent_class'] = BCAgent ## look in here and implement this
        self.params['agent_params'] = agent_params

        ################
        ## RL TRAINER
        ################

        self.rl_trainer = RL_Trainer(self.params) ## look in here and implement this

        #######################
        ## LOAD EXPERT POLICY
        #######################

        print('Loading expert policy from...', self.params['expert_policy_file'])
        self.loaded_expert_policy = LoadedGaussianPolicy(self.params['expert_policy_file'])
        print('Done restoring expert policy...')

    def run_training_loop(self):

        self.rl_trainer.run_training_loop(
            n_iter=self.params['n_iter'],
            initial_expertdata=self.params['expert_data'],
            collect_policy=self.rl_trainer.agent.actor,
            eval_policy=self.rl_trainer.agent.actor,
            relabel_with_expert=self.params['do_dagger'],
            expert_policy=self.loaded_expert_policy,
        )


In [12]:
!rm -rf /content/hw_16831/hw1/data/

In [25]:
#@title create directory for logging

if args.do_dagger:
    logdir_prefix = 'q2_'
    assert args.n_iter>1, ('DAgger needs more than 1 iteration (n_iter>1) of training, to iteratively query the expert and train (after 1st warmstarting from behavior cloning).')
else:
    logdir_prefix = 'q1_'
    assert args.n_iter==1, ('Vanilla behavior cloning collects expert data just once (n_iter=1)')

data_path ='/content/hw_16831/hw1/data'
if not (os.path.exists(data_path)):
    os.makedirs(data_path)
logdir = logdir_prefix + args.exp_name + '_' + args.env_name + \
         '_' + time.strftime("%d-%m-%Y_%H-%M-%S")
logdir = os.path.join(data_path, logdir)
args['logdir'] = logdir
if not(os.path.exists(logdir)):
    os.makedirs(logdir)

In [26]:
# Make sure you're in the 16831-F25-HW/hw1 folder when running this.
## run training
print(args.logdir)
trainer = BC_Trainer(args)
trainer.run_training_loop()

/content/hw_16831/hw1/data/q2_test_bc_humanoid_dagger_Humanoid-v2_05-02-2026_02-39-46
########################
logging outputs to  /content/hw_16831/hw1/data/q2_test_bc_humanoid_dagger_Humanoid-v2_05-02-2026_02-39-46
########################
Using GPU id 0
Loading expert policy from... /content/hw_16831/16831-S26-HW/hw1/rob831/policies/experts/Humanoid.pkl


/usr/local/lib/python3.12/dist-packages/gym/envs/registration.py:593: UserWarning: WARN: The environment Humanoid-v2 is out of date. You should consider upgrading to version `v4`.
  logger.warn(


obs (1, 376) (1, 376)
Done restoring expert policy...


********** Iteration 0 ************

Training agent using sampled data from replay buffer...

Beginning logging procedure...



INFO:tensorboardX.summary:Summary name Training Loss is illegal; using Training_Loss instead.


Eval_AverageReturn : 275.76593017578125
Eval_StdReturn : 22.548099517822266
Eval_MaxReturn : 414.0909118652344
Eval_MinReturn : 232.46231079101562
Eval_AverageEpLen : 50.79187817258883
Train_AverageReturn : 10344.517578125
Train_StdReturn : 20.9814453125
Train_MaxReturn : 10365.4990234375
Train_MinReturn : 10323.5361328125
Train_AverageEpLen : 1000.0
Train_EnvstepsSoFar : 0
TimeSinceStart : 30.73403525352478
Training Loss : 0.10283631831407547
Initial_DataCollection_AverageReturn : 10344.517578125
Done logging...




********** Iteration 1 ************


Relabelling collected observations with labels from an expert policy...

Training agent using sampled data from replay buffer...

Beginning logging procedure...



...

Eval_AverageReturn : 4436.51220703125
Eval_StdReturn : 3423.852294921875
Eval_MaxReturn : 10386.4619140625
Eval_MinReturn : 451.7569580078125
Eval_AverageEpLen : 456.72727272727275
Train_AverageReturn : 6333.5380859375
Train_StdReturn : 3736.0244140625
Train_MaxReturn : 10069.5625
Train_MinReturn : 2597.513427734375
Train_AverageEpLen : 647.0
Train_EnvstepsSoFar : 117203
TimeSinceStart : 3232.8273935317993
Training Loss : 0.14107118546962738
Initial_DataCollection_AverageReturn : 10344.517578125
Done logging...




********** Iteration 98 ************


Relabelling collected observations with labels from an expert policy...

Training agent using sampled data from replay buffer...

Beginning logging procedure...



INFO:tensorboardX.summary:Summary name Training Loss is illegal; using Training_Loss instead.


Eval_AverageReturn : 4315.5283203125
Eval_StdReturn : 3037.5029296875
Eval_MaxReturn : 10156.87890625
Eval_MinReturn : 468.9561462402344
Eval_AverageEpLen : 452.0869565217391
Train_AverageReturn : 10412.546875
Train_StdReturn : 0.0
Train_MaxReturn : 10412.546875
Train_MinReturn : 10412.546875
Train_AverageEpLen : 1000.0
Train_EnvstepsSoFar : 118203
TimeSinceStart : 3270.229919195175
Training Loss : 0.14368075132369995
Initial_DataCollection_AverageReturn : 10344.517578125
Done logging...




********** Iteration 99 ************


Relabelling collected observations with labels from an expert policy...

Training agent using sampled data from replay buffer...

Beginning logging procedure...



INFO:tensorboardX.summary:Summary name Training Loss is illegal; using Training_Loss instead.


Eval_AverageReturn : 4644.36328125
Eval_StdReturn : 3385.692626953125
Eval_MaxReturn : 10365.142578125
Eval_MinReturn : 520.0051879882812
Eval_AverageEpLen : 471.2608695652174
Train_AverageReturn : 4894.6171875
Train_StdReturn : 760.7511596679688
Train_MaxReturn : 5763.02880859375
Train_MinReturn : 3910.40283203125
Train_AverageEpLen : 503.0
Train_EnvstepsSoFar : 119712
TimeSinceStart : 3308.818233013153
Training Loss : 0.15887480974197388
Initial_DataCollection_AverageReturn : 10344.517578125
Done logging...




In [ ]:
# !pip install --upgrade tensorboard

In [ ]:
#@markdown You can visualize your runs with tensorboard from within the notebook

# %load_ext tensorboard
# %tensorboard --logdir /content/hw_16831/hw1/data

## Running DAgger (Problem 2)
Modify the settings above:
1. check the `do_dagger` box
2. set `n_iters` to `10`
and then rerun the code.